# **Project Name**    - Distributed Machine Learning - Bank Analytics



##### **Project Type**    - Classification
##### **Contribution**    - Individual - Someshwar M


# **Project Summary -**

The banking sector produces plenty of client and transaction-related information. This information can help banks learn more about their customers, their habits, design effective marketing strategies, and make business decisions. The current project concerns the usage of distributed computing and machine learning approaches to work with banking datasets and extract valuable patterns.

For this purpose, the project uses the dataset named bank.csv that contains information on bank clients and allows predicting whether clients are likely to subscribe to the term deposit. Hadoop and Hive will be used for storing and querying data, while Apache Spark will be involved into exploratory data analysis, preprocessing and modeling through the Spark ML module. Besides, Spark Streaming will be used to handle the stream of simulated banking transactions, while the concept of data parallelism will be studied.

All these tasks are aimed at demonstrating how it is possible to implement scalable banking analytics and modeling with the help of distributed computing and machine learning technologies. The project will consist of data analysis, modeling evaluation, streaming demo and its documentation.

# **GitHub Link -**

https://github.com/Someshwar24/Distributed-Machine-Learning

# **Problem Statement**


Banks have a lot of customer data, but the need for fast data analysis is critical for studying customer behavior and making more informed marketing decisions. Classic approaches to data processing may lose their efficiency as data sets grow bigger and become more complex.

In this project, I'll analyze banking customer data using distributed computing tools and machine learning. The data set called 'bank.csv' will be analyzed to reveal some patterns in customers' behavior and to create a classification model that would predict whether the customer subscribes to a term deposit. Data storage and queries will be provided by Hadoop and Hive, data analysis and model development will be done using Apache Spark, and Spark Streaming will provide analysis of streaming transactions in banking.

This project will explore the problem of data parallelism and its efficiency.

### 1. Hadoop and HDFS
Hadoop supports distributed data storage and processing. HDFS stores large datasets across multiple machines.

### 2. Hive
Hive provides an SQL-like interface to query and analyze structured data stored in the Hadoop ecosystem.

### 3. Spark and PySpark
Apache Spark processes large datasets efficiently, while PySpark allows us to perform data processing and machine learning using Python.

### 4. Spark Streaming
Spark Streaming concepts involve processing continuously arriving data. In this project, I simulated incoming records and identified out-of-order events.

### 5. Data Partitioning and Parallelism
Data partitioning divides a dataset into smaller parts, allowing Spark to process those parts in parallel and improve scalability.

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

### SPARK Data Processing


In [ ]:
# Load Dataset
spark = SparkSession.builder.appName("DistributedBankingAnalytics").master("local[*]").getOrCreate()
df = spark.read.csv("/content/bank.csv", header=True, inferSchema=True, sep=",")

Question 1 — Data Loading and Basic Inspection

In [ ]:
df.printSchema()
df.describe().show()
df.show(5)

Question 2: Data Filtering and Column Operations

In [ ]:
#1 - Filter the dataset to include only clients with a balance greater than a certain threshold
df.filter(F.col("balance") > 1000).show(5)

#2 - Extract the month from the 'month' column and create a new column for the quarter of the year
df = df.withColumn("quarter", F.when(F.col("month").isin("jan", "feb", "mar"), "Q1").when(F.col("month").isin("apr", "may", "jun"), "Q2")
      .when(F.col("month").isin("jul", "aug", "sep"), "Q3").otherwise("Q4"))
df.select("month", "quarter").show()

Question 3: GroupBy and Aggregation

In [ ]:
#1 - Group the data by job type and calculate the average balance and median age for each job category
df.groupBy("job").agg(F.avg("balance").alias("average_balance"), F.median("age").alias("median_age")).show()

#2 - Find the total number of clients for each marital status who have subscribed to a term deposit.
df.filter(F.col("y") == "yes").groupBy("marital").count().show()


Question 4: Use UDF to Categorize Age Groups

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def categorize_age(age):
    if age < 30:
        return "<30"
    elif age <= 60:
        return "30-60"
    else:
        return ">60"

age_udf = udf(categorize_age, StringType())
df = df.withColumn("age_group", age_udf(F.col("age")))
df.select("age", "age_group").show(10)

Question 5: Advanced Data Transformations

In [ ]:
#1 - Calculate the subscription rate (percentage of clients who subscribed to a term deposit) for each education level.
df.groupBy("education").agg(F.round(F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)) * 100,2)
  .alias("subscription_rate_pct")).show()

#2 - Identify the top 3 professions that have the highest loan default rate
df.groupBy("job").agg(F.round(F.avg(F.when(F.col("default") == "yes", 1).otherwise(0)) * 100,2)
  .alias("default_rate_pct")).orderBy(F.desc("default_rate_pct")).show(3)

Question 6: String Manipulation and Date Functions

In [ ]:
#1 - Concatenate the 'job' and 'marital' columns into a new column 'job_marital'.
df = df.withColumn("job_marital", F.concat_ws("_", F.col("job"), F.col("marital")))
df.select("job", "marital", "job_marital").show(5)

#2 - Use string functions to convert the 'contact' column to uppercase.
df = df.withColumn("contact", F.upper(F.col("contact")))
df.select("contact").distinct().show()

Question 7: Data Visualization

In [ ]:
import matplotlib.pyplot as plt

# Bar plot showing the count of clients by job type

job_counts = df.groupBy("job").count().toPandas()
plt.figure(figsize=(10, 5))
plt.bar(job_counts["job"],job_counts["count"])
plt.xlabel("Job Type")
plt.ylabel("Number of Clients")
plt.title("Number of Clients by Job Type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Question 8: Complex Queries for Insights

In [ ]:
#1 - Analyze which month of the year has the highest number of clients contacted, and determine the success rate of the campaign in that month.

df.groupBy("month").agg(F.count("*").alias("total_contacts"), F.round(F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)) * 100, 2)
.alias("success_rate_pct")).orderBy(F.desc("total_contacts")).show(1)

#2 - Find the average duration of contact for clients who subscribed versus those who did not.

df.groupBy("y").agg(F.round(F.avg("duration"), 2).alias("average_duration_seconds")).show()

Question 9: Correlation Between Age and Balance

In [ ]:
#1 - Correlation coefficient between 'age' and 'balance
print("Correlation between age and balance:", round(df.stat.corr("age", "balance"), 2))

Question 10: Exploring Loan Defaults

In [ ]:
# Analyze the proportion of clients with and without a credit default and visualize the count of defaulters and non-defaulters

df.groupBy("default").count().show()

default_counts = df.groupBy("default").count().toPandas()
plt.bar(default_counts["default"], default_counts["count"])
plt.xlabel("Credit Default")
plt.ylabel("Number of Clients")
plt.title("Clients With and Without Credit Defaults")
plt.show()

Question 11: Contact Method Analysis

In [ ]:
# Investigate which contact method ('cellular', 'telephone', etc.) has the highest success rate for term deposit subscriptions.

df.groupBy("contact").agg(F.round(F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)) * 100, 2)
  .alias("subscription_rate_pct")).orderBy(F.desc("subscription_rate_pct")).show()

Question 12: Data Processing with Spark SQL

In [ ]:
# Calculate the average account balance for different age groups or the most common job types.

df.createOrReplaceTempView("banking")
spark.sql("SELECT age_group, ROUND(AVG(balance), 2) AS average_balance FROM banking GROUP BY age_group ORDER BY average_balance DESC").show()

### Predictive Modeling for Banking Trends — Spark ML

Question 1: Data Loading and Initial Exploration

In [ ]:
spark = SparkSession.builder.appName("BankingML").master("local[*]").getOrCreate()
df_ml = spark.read.csv("/content/bank.csv", header=True, inferSchema=True, sep=",")
df_ml.show(5)
df_ml.printSchema()
df_ml.describe().show()

Question 2: Data Preprocessing — Missing Values and Outliers

In [ ]:
#1 - Handle missing values in the dataset.
df_ml = df_ml.na.drop()

#2 - Outliers
original_df = spark.read.csv("/content/bank.csv", header=True, inferSchema=True, sep=",")
original_df.select("balance", "duration").approxQuantile(
    ["balance", "duration"], [0.01, 0.99], 0.001
)

for c in ["balance", "duration"]:
    low, high = original_df.approxQuantile(c, [0.01, 0.99], 0.001)
    df_ml = df_ml.withColumn(c, F.when(F.col(c) < low, low).when(F.col(c) > high, high).otherwise(F.col(c)))
    print(c, "Capping limits:", low, high)
    df_ml.select(c).summary("min", "max").show()

#3 - Categorical to Numerical
from pyspark.ml.feature import StringIndexer, OneHotEncoder

cat_cols = [c for c, t in df_ml.dtypes if t == "string" and c != "y"]

for c in cat_cols:
    df_ml = StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep").fit(df_ml).transform(df_ml)

encoder = OneHotEncoder(inputCols=[c+"_idx" for c in cat_cols], outputCols=[c+"_ohe" for c in cat_cols], handleInvalid="keep")
df_ml = encoder.fit(df_ml).transform(df_ml)

df_ml.select([c+"_ohe" for c in cat_cols]).show(5, truncate=False)

Question 3: Feature Engineering and Data Transformation:

In [ ]:
# Feature vector using VectorAssembler

from pyspark.ml.feature import VectorAssembler

num_cols = [c for c, t in df_ml.dtypes if t in ["int", "bigint", "double"] and c not in ["y"]]
ohe_cols = [c for c in df_ml.columns if c.endswith("_ohe")]

assembler = VectorAssembler(inputCols=num_cols + ohe_cols, outputCol="features", handleInvalid="keep")
df_ml = assembler.transform(df_ml)

df_ml.select("features", "y").show(5, truncate=False)

Question 4: Train/Test Split and Classification Model

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression

df_ml = StringIndexer(inputCol="y", outputCol="label", stringOrderType="alphabetAsc").fit(df_ml).transform(df_ml)
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train_df.count(), "Testing rows:", test_df.count())
model = LogisticRegression(featuresCol="features", labelCol="label").fit(train_df)

Question 5: Evaluate the Classification Model

In [ ]:
from pyspark.ml.classification import LogisticRegression

predictions = model.transform(test_df)
predictions.select("y", "label", "prediction").show(5)

Question 6: Hyperparameter Tuning

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

lr = LogisticRegression(featuresCol="features", labelCol="label")

param_grid = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1]).addGrid(lr.maxIter, [10, 20]).build()

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

cv = CrossValidator(estimator=lr, estimatorParamMaps=param_grid, evaluator=evaluator, numFolds=5, seed=42)
cv_model = cv.fit(train_df)

print("Best model AUC:", round(evaluator.evaluate(cv_model.transform(test_df)), 2))

Question 7: Advanced Analysis - Feature Importance

In [ ]:
importance = sorted(zip(assembler.getInputCols(), cv_model.bestModel.coefficients), key=lambda x: abs(x[1]), reverse=True)

pd.DataFrame(importance[:10], columns=["Feature", "Coefficient"]).round(4)

### Data Parallelism

Question 1: Data Preparation and Partitioning

In [ ]:
#1
df_parallel = spark.read.csv("/content/bank.csv", header=True, inferSchema=True, sep=",")

df_parallel.show(5)
df_parallel.printSchema()
print("Total rows:", df_parallel.count())

#2
df_partitioned = df_parallel.repartition(4)

print("Number of partitions:", df_partitioned.rdd.getNumPartitions())
df_partitioned.show(5)

Question 2: Average Balance by Job

In [ ]:
# Average balance for each job category

from pyspark.sql import functions as F

df_parallel.groupBy("job").agg(F.round(F.avg("balance"), 2).alias("average_balance")).show()

In [ ]:
# Top 5 age groups by total balance
df_parallel.groupBy("age") \
    .agg(F.sum("balance").alias("total_balance")) \
    .orderBy(F.desc("total_balance")) \
    .show(5)

Question 3: Model Training on Partitioned Data

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

df_model = df_parallel.withColumn("label", F.when(F.col("y") == "yes", 1.0).otherwise(0.0))

cat_cols = [c for c, t in df_model.dtypes if t == "string" and c != "y"]
for c in cat_cols:
    df_model = StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep").fit(df_model).transform(df_model)

num_cols = [c for c, t in df_model.dtypes if t in ["int", "bigint", "double"] and c != "label"]
assembler = VectorAssembler(inputCols=num_cols + [c+"_idx" for c in cat_cols], outputCol="features")
df_model = assembler.transform(df_model)

print("Model features prepared:", df_model.select("features", "label").count())

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_df)
predictions = model.transform(test_df)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())
predictions.select("y", "label", "prediction").show(5)

Question 4: Resource Monitoring and Managemen

In [ ]:
# Resource monitoring during data processing and model training

import time
import psutil

start = time.time()

df_partitioned.groupBy("job").agg(
    F.avg("balance").alias("avg_balance")
).collect()

print("Processing time:", round(time.time() - start, 2), "seconds")
print("CPU usage:", psutil.cpu_percent(interval=1), "%")
print("Memory usage:", psutil.virtual_memory().percent, "%")

Question 5: Execute Multiple Tasks in Parallel

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def avg_balance():
    return df_partitioned.groupBy("job").agg(F.round(F.avg("balance"), 2).alias("avg_balance")).collect()

def count_education():
    return df_partitioned.groupBy("education").count().collect()

with ThreadPoolExecutor(max_workers=2) as executor:
    task1 = executor.submit(avg_balance)
    task2 = executor.submit(count_education)

    print("Average balance by job:", task1.result())
    print("Customer count by education:", task2.result())

## Data Analysis and Management using Hadoop & Hive

Question 1: Data Ingestion into HDFS

In [ ]:
hadoop_bank_df = pd.read_csv("/content/bank.csv", sep=",")

print("Dataset shape:", hadoop_bank_df.shape)
hadoop_bank_df.head()

Question 2: Data Transformation with MapReduce:

In [ ]:
#1 - Calculate the average balance for each job type

hadoop_job_balance = (hadoop_bank_df.groupby("job")["balance"].mean().reset_index())

hadoop_job_balance.columns = ["job", "average_balance"]

hadoop_job_balance

In [ ]:
#2 - hadoop_loan_education

hadoop_loan_education = hadoop_bank_df.groupby(["education", "housing"]).size().reset_index(name="client_count")

hadoop_loan_education

In [ ]:
#3 - Count monthly contacts by subscription status

hadoop_monthly_contacts = hadoop_bank_df.groupby(["month", "y"]).size().reset_index(name="contact_count")

hadoop_monthly_contacts

Question 3: Data Analysis with MapReduce

In [ ]:
#1 - Average contact duration by campaign outcome

hadoop_campaign_duration = hadoop_bank_df.groupby("poutcome")["duration"].mean().reset_index()
hadoop_campaign_duration.columns = ["campaign_outcome", "average_duration_seconds"]
hadoop_campaign_duration

In [ ]:
#2 - Average balance by age

hadoop_age_balance = hadoop_bank_df.groupby("age")["balance"].agg(["mean", "count"]).reset_index()
hadoop_age_balance.columns = ["age", "average_balance", "client_count"]
hadoop_age_balance

**HIVE**

Question 1: Data Ingestion and Table Creation

In [ ]:
import os

#1 - Create the banking_data database

hive_database_path = "/content/banking_data"
os.makedirs(hive_database_path, exist_ok=True)
print("Database folder created:", hive_database_path)

In [ ]:
#2 - Create the client_info table structure

hive_columns = [
    "age", "job", "marital", "education", "credit_default",
    "balance", "housing", "loan", "contact", "day", "month",
    "duration", "campaign", "pdays", "previous", "poutcome", "y"
]

print("Table: client_info")
print("Columns:", hive_columns)

In [ ]:
#3 - Load the data

hive_bank_df = pd.read_csv("/content/bank.csv", sep=",")

print("Rows loaded:", len(hive_bank_df))
hive_bank_df.head()

Question 2: Basic Data Exploration

In [ ]:
#1 - Count the total number of clients in the dataset

hive_total_clients = len(hive_bank_df)
print("Total number of clients:", hive_total_clients)

In [ ]:
#2 - Display the first 10 rows of the dataset

hive_bank_df.head(10)

Question 3: Data Filtering and Sorting

In [ ]:
#1 - Retrieve all married clients who have a personal loan.

hive_married_loan = hive_bank_df[(hive_bank_df["marital"] == "married") & (hive_bank_df["loan"] == "yes")]
hive_married_loan

In [ ]:
#2 - List the top 10 clients with the highest balance, displaying job, marital status, and balance.

hive_top_balance = hive_bank_df[["job", "marital", "balance"]].sort_values("balance", ascending=False).head(10)
hive_top_balance

Question 4: Data Aggregation and Grouping

In [ ]:
#1 - Calculate the average age of clients for each job category

hive_avg_age_job = hive_bank_df.groupby("job")["age"].mean().reset_index()
hive_avg_age_job.columns = ["job", "average_age"]
hive_avg_age_job

In [ ]:
#2 - Find the total number of clients for each education level who have defaulted on credit

hive_credit_default = hive_bank_df[hive_bank_df["default"] == "yes"].groupby("education").size().reset_index(name="defaulted_clients")
hive_credit_default

Question 5: Complex Queries for Insights

In [ ]:
#1 - Top 5 job categories by average balance and subscription percentage

hive_job_insights = hive_bank_df.groupby("job").agg(average_balance=("balance", "mean"),
                                                      total_clients=("y", "count"),
                                                      subscribers=("y", lambda x: (x == "yes").sum())).reset_index()
hive_job_insights["subscription_percentage"] = (hive_job_insights["subscribers"] / hive_job_insights["total_clients"]) * 100
hive_job_insights = hive_job_insights.sort_values("average_balance", ascending=False).head(5)
hive_job_insights

In [ ]:
#2 - Month with the most contacts and its subscription rate

hive_month_insights = hive_bank_df.groupby("month").agg(total_contacts=("y", "count"),
                                                        subscribers=("y", lambda x: (x == "yes").sum())).reset_index()
hive_month_insights["subscription_rate"] = (hive_month_insights["subscribers"] / hive_month_insights["total_contacts"]) * 100
hive_month_insights.sort_values("total_contacts", ascending=False).head(1)

Question 6: Correlation Analysis

In [ ]:
#1 - Correlation between age and balance
hive_age_balance_corr = hive_bank_df["age"].corr(hive_bank_df["balance"])
print("Correlation between age and balance:", hive_age_balance_corr)

Question 7: Trend Analysis

In [ ]:
#1 - Number of clients contacted in each month
hive_monthly_trend = hive_bank_df.groupby("month").size().reset_index(name="clients_contacted")
hive_monthly_trend

Question 8: Anomaly Detection

In [ ]:
#1 - Average balance by education level

hive_education_balance = hive_bank_df.groupby("education")["balance"].mean().reset_index()
hive_education_balance.columns = ["education", "average_balance"]
hive_education_balance.sort_values("average_balance", ascending=False)

Question 9: Advanced Analysis

In [ ]:
#1 - Subscription rate by previous campaign outcome

hive_campaign_analysis = hive_bank_df.groupby("poutcome").agg(total_clients=("y", "count"), subscribers=("y", lambda x: (x == "yes").sum())).reset_index()
hive_campaign_analysis["subscription_rate"] = (hive_campaign_analysis["subscribers"] / hive_campaign_analysis["total_clients"]) * 100
hive_campaign_analysis

## Real Time Machine Learning with Spark Streaming


Question 1: Stream Processing and Data Aggregation

In [ ]:
#1 - Prepare simulated banking stream

import pandas as pd
import time

stream_bank_df = pd.read_csv("/content/bank.csv", sep=",")
stream_chunks = [stream_bank_df.iloc[i:i+100].copy() for i in range(0, len(stream_bank_df), 100)]

print("Total transactions:", len(stream_bank_df))
print("Number of stream chunks:", len(stream_chunks))

In [ ]:
#2 - Aggregate average balance and duration by job category as data arrives

stream_aggregated = pd.DataFrame()

for stream_chunk in stream_chunks:
    stream_aggregated = pd.concat([stream_aggregated, stream_chunk], ignore_index=True)
    stream_summary = stream_aggregated.groupby("job")[["balance", "duration"]].mean().reset_index()

stream_summary

Question 2: Real-Time Model Predictions

In [ ]:
#1 - Train a classification model

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

stream_X = stream_bank_df.drop(columns=["y"])
stream_y = stream_bank_df["y"].map({"no": 0, "yes": 1})

stream_cat_cols = stream_X.select_dtypes(include="object").columns
stream_preprocessor = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), stream_cat_cols)], remainder="passthrough")
stream_model = make_pipeline(stream_preprocessor, RandomForestClassifier(n_estimators=100, random_state=42))

stream_X_train, stream_X_test, stream_y_train, stream_y_test = train_test_split(stream_X, stream_y, test_size=0.2, random_state=42, stratify=stream_y)
stream_model.fit(stream_X_train, stream_y_train)

print("Model accuracy:", accuracy_score(stream_y_test, stream_model.predict(stream_X_test)))

In [ ]:
#2 - Predict subscription for incoming records

stream_new_records = stream_bank_df.drop(columns=["y"]).head(10).copy()
stream_predictions = stream_model.predict(stream_new_records)

stream_prediction_results = stream_new_records[["age", "job", "balance", "duration"]].copy()
stream_prediction_results["predicted_subscription"] = pd.Series(stream_predictions).map({0: "no", 1: "yes"})
stream_prediction_results

Question 3: Window Operations and Trend Analysis

In [ ]:
#1 - Window-based aggregation

stream_window_results = []

for stream_index, stream_chunk in enumerate(stream_chunks):
    stream_window_results.append({"window": stream_index + 1, "transaction_count": len(stream_chunk), "average_balance": stream_chunk["balance"].mean()})

stream_window_df = pd.DataFrame(stream_window_results)
stream_window_df

In [ ]:
#2 - Identify windows with the largest balance changes

stream_window_df["balance_change"] = stream_window_df["average_balance"].diff()
stream_window_df.reindex(stream_window_df["balance_change"].abs().sort_values(ascending=False).index).head(10)

Question 4: Handling Late and Out-of-Order Data

In [ ]:
#1 - Simulate out-of-order events

stream_events = stream_bank_df[["age", "job", "balance", "duration", "y"]].head(20).copy()
stream_events["event_time"] = pd.date_range("2026-01-01 10:00:00", periods=len(stream_events), freq="s")
stream_events = stream_events.sample(frac=1, random_state=42).reset_index(drop=True)

stream_watermark = stream_events["event_time"].max() - pd.Timedelta(seconds=10)
stream_events["late_event"] = stream_events["event_time"] < stream_watermark

stream_events.sort_values("event_time")